# Day 4: Spark SQL and Complex Joins Solutions

## Welcome!
Use these solutions to check your work from `04_exercise.ipynb`.

## Before You Start
- Run `docker-compose up` in `01_basic_spark`.
- Open Jupyter at `http://localhost:8888`.
- Use `covid-data.csv` in `covid-dataset/`.
- Please refer [week-2 materials](https://github.com/LD-LINC/Week2---SQL-And-Data-Modeling) and [exercises](https://github.com/LD-LINC/Week2---SQL-And-Data-Modeling/tree/main/05_weekly_project) for better understanding of SQL.
- Download the dataset if needed: https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv

## Solutions

### Exercise 1: Create a Temporary View
#### What to Do
- Load `covid-data.csv` into a DataFrame and create a temporary view named `covid_view`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession
spark = SparkSession.builder.appName("SparkSQL1").getOrCreate()  # Shorter app name

# Read CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Create temporary SQL view
df.createOrReplaceTempView("covid_view")  # Enables SQL queries

# Show first 5 rows
df.show(5)


**Expected Output**:
```
+--------+---------+-----------+----------+-----------+---------+------------------+------------+----------+...
|iso_code|continent|   location|      date|total_cases|new_cases|new_cases_smoothed|total_deaths|new_deaths|...
+--------+---------+-----------+----------+-----------+---------+------------------+------------+----------+...
|     AFG|     Asia|Afghanistan|2020-01-05|          0|        0|              NULL|           0|         0|...
|     AFG|     Asia|Afghanistan|2020-01-06|          0|        0|              NULL|           0|         0|...
|     AFG|     Asia|Afghanistan|2020-01-07|          0|        0|              NULL|           0|         0|...
|     AFG|     Asia|Afghanistan|2020-01-08|          0|        0|              NULL|           0|         0|...
|     AFG|     Asia|Afghanistan|2020-01-09|          0|        0|              NULL|           0|         0|...
+--------+---------+-----------+----------+-----------+---------+------------------+------------+----------+...
```

### Exercise 2: Basic SQL Query
#### What to Do
- Select `location`, `date`, and `new_cases` where `new_cases` > 1000.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession
spark = SparkSession.builder.appName("SparkSQL2").getOrCreate()  # Shorter app name

# Read CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Filter out rows with null continent
df = df.filter(col("continent").isNotNull())

# Create temporary SQL view
df.createOrReplaceTempView("covid_view")  # Enables SQL queries

# Run SQL query for new_cases > 1000
result = spark.sql("""
    SELECT location, date, new_cases 
    FROM covid_view 
    WHERE new_cases > 1000 AND new_cases IS NOT NULL
""")

# Show first 10 rows
result.show(10)


**Expected Output**:
```
+-----------+----------+---------+
|   location|      date|new_cases|
+-----------+----------+---------+
|Afghanistan|2020-05-10|     1392|
|Afghanistan|2020-05-17|     2490|
|Afghanistan|2020-05-24|     3813|
|Afghanistan|2020-05-31|     4577|
|Afghanistan|2020-06-07|     5108|
|Afghanistan|2020-06-14|     4551|
|Afghanistan|2020-06-21|     4195|
|Afghanistan|2020-06-28|     2319|
|Afghanistan|2020-07-05|     2056|
|Afghanistan|2020-07-12|     1679|
+-----------+----------+---------+
```

### Exercise 3: Group By with SQL
#### What to Do
- Calculate the total `new_cases` per `continent` using Spark SQL.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession
spark = SparkSession.builder.appName("SparkSQL3").getOrCreate()  # Shorter app name

# Read CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Filter out rows with null continent
df = df.filter(col("continent").isNotNull())

# Create temporary SQL view
df.createOrReplaceTempView("covid_view")  # Enables SQL queries

# SQL query: sum of new_cases per continent
result = spark.sql("""
    SELECT continent, SUM(new_cases) AS total_new_cases
    FROM covid_view
    WHERE continent IS NOT NULL
    GROUP BY continent
""")

# Show results
result.show()


**Expected Output**:
```
+-------------+---------------+
|    continent|total_new_cases|
+-------------+---------------+
|       Europe|      252916868|
|       Africa|       13146831|
|North America|      124492698|
|South America|       68811012|
|      Oceania|       15003468|
|         Asia|      301564180|
+-------------+---------------+
```

### Exercise 4: Filter with WHERE Clause
#### What to Do
- Find records where `reproduction_rate` > 1.2 and `continent` is not null, showing `location`, `date`, `reproduction_rate`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession with shorter app name
spark = SparkSession.builder.appName("SparkSQL4").getOrCreate()

# Read CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Filter rows where continent is not null
df = df.filter(col("continent").isNotNull())

# Create temporary SQL view
df.createOrReplaceTempView("covid_view")  # Enables running SQL queries

# SQL query: select locations and dates where reproduction_rate > 1.2
result = spark.sql("""
    SELECT location, date, reproduction_rate
    FROM covid_view
    WHERE reproduction_rate > 1.2 AND continent IS NOT NULL
""")

# Show first 10 rows
result.show(10)


**Expected Output**:
```
+-----------+----------+-----------------+
|   location|      date|reproduction_rate|
+-----------+----------+-----------------+
|Afghanistan|2020-03-29|             1.51|
|Afghanistan|2020-03-30|             1.51|
|Afghanistan|2020-03-31|             1.52|
|Afghanistan|2020-04-01|             1.51|
|Afghanistan|2020-04-02|             1.51|
|Afghanistan|2020-04-03|              1.5|
|Afghanistan|2020-04-04|             1.49|
|Afghanistan|2020-04-05|             1.49|
|Afghanistan|2020-04-06|             1.49|
|Afghanistan|2020-04-07|             1.49|
+-----------+----------+-----------------+
```

### Exercise 5: Average KPI with SQL
#### What to Do
- Calculate the average `total_cases_per_million` per `continent` using Spark SQL, sorted descending.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession with shorter app name
spark = SparkSession.builder.appName("SparkSQL5").getOrCreate()

# Read CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Filter rows where continent is not null
df = df.filter(col("continent").isNotNull())

# Create temporary SQL view
df.createOrReplaceTempView("covid_view")  # Enables running SQL queries

# SQL query: calculate average total_cases_per_million per continent
result = spark.sql("""
    SELECT continent, AVG(total_cases_per_million) AS avg_cases_per_million
    FROM covid_view
    WHERE continent IS NOT NULL
    GROUP BY continent
    ORDER BY avg_cases_per_million DESC
""")

# Show result
result.show()


**Expected Output**:
```
+-------------+---------------------+
|    continent|avg_cases_per_million|
+-------------+---------------------+
|       Europe|   224006.97074086822|
|North America|   132425.64220693253|
|      Oceania|   113796.86926000904|
|South America|     111028.525150198|
|         Asia|    80234.34640331697|
|       Africa|   26604.429582991594|
+-------------+---------------------+
```

### Exercise 6: HAVING Clause
#### What to Do
- Find continents with average `total_deaths_per_million` > 500, showing only `continent` and the average.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession with shorter app name
spark = SparkSession.builder.appName("SparkSQL6").getOrCreate()

# Read CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Filter rows where continent is not null
df = df.filter(col("continent").isNotNull())

# Create temporary SQL view
df.createOrReplaceTempView("covid_view")  # Enables SQL queries on DataFrame

# SQL query: average total_deaths_per_million per continent, only include continents with avg > 500
result = spark.sql("""
    SELECT continent, AVG(total_deaths_per_million) AS avg_deaths_per_million
    FROM covid_view
    WHERE total_deaths_per_million IS NOT NULL
    GROUP BY continent
    HAVING AVG(total_deaths_per_million) > 500
""")

# Show result
result.show()


**Expected Output**:
```
+-------------+----------------------+
|    continent|avg_deaths_per_million|
+-------------+----------------------+
|       Europe|    1758.8532228781592|
|North America|      967.805120057099|
|South America|     1687.418800989931|
+-------------+----------------------+
```

### Exercise 7: Inner Join with SQL
#### What to Do
- Create two views from `covid_view`: one for 2020 data and one for 2021 data. Perform an inner join on `location` where `total_cases` > 100000 in both years.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession with shorter app name
spark = SparkSession.builder.appName("SparkSQL7").getOrCreate()

# Read CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("data/owid-covid-data.csv")

# Filter rows where continent is not null
df = df.filter(col("continent").isNotNull())

# Create temporary SQL view for general queries
df.createOrReplaceTempView("covid_view")  # Enables SQL queries on DataFrame

# Create temp view for 2020 with locations having total_cases > 100k
spark.sql("""
CREATE OR REPLACE TEMP VIEW covid_2020 AS
SELECT location, MAX(total_cases) AS cases_2020
FROM covid_view
WHERE YEAR(date) = 2020 AND total_cases > 100000
GROUP BY location
""")

# Create temp view for 2021 with locations having total_cases > 100k
spark.sql("""
CREATE OR REPLACE TEMP VIEW covid_2021 AS
SELECT location, MAX(total_cases) AS cases_2021
FROM covid_view
WHERE YEAR(date) = 2021 AND total_cases > 100000
GROUP BY location
""")

# Join 2020 and 2021 views to compare cases
result = spark.sql("""
SELECT c20.location, c20.cases_2020, c21.cases_2021
FROM covid_2020 c20
INNER JOIN covid_2021 c21
ON c20.location = c21.location
""")

# Show top 10 results
result.show(10)


+--------------------+----------+----------+
|            location|cases_2020|cases_2021|
+--------------------+----------+----------+
|           Argentina|   1629908|   5559916|
|          Azerbaijan|    211764|    614119|
|             Armenia|    157834|    344481|
|             Austria|    344732|   1252088|
|             Belgium|    638760|   2048110|
|             Belarus|    184922|    692601|
|             Bolivia|    153590|    575247|
|          Bangladesh|    509148|   1583253|
|              Brazil|   7448560|  22230737|
|Bosnia and Herzeg...|    109330|    287716|
+--------------------+----------+----------+
only showing top 10 rows



**Expected Output**:
```
+----------+----------+----------+
|  location|cases_2020|cases_2021|
+----------+----------+----------+
| Argentina|   1629908|   5559916|
|   Belgium|    638760|   2048110|
|   Ecuador|    209274|    541368|
|   Belarus|    184922|    692601|
|     Chile|    598394|   1799125|
|   Croatia|    204312|    693102|
|   Bolivia|    153590|    575247|
|   Czechia|    675062|   2486451|
|   Denmark|    151167|    697563|
|Bangladesh|    509148|   1583253|
+----------+----------+----------+
```

### Exercise 8: Left Join with SQL
#### What to Do
- Perform a left join between `covid_view` and a view of locations with `new_deaths` > 1000, showing `location`, `date`, and `new_deaths`.

In [41]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession with shorter app name
spark = SparkSession.builder.appName("SparkSQL8").getOrCreate()

# Read CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("data/owid-covid-data.csv")

# Filter rows with non-null continent
df = df.filter(col("continent").isNotNull())

# Create temporary view for SQL queries
df.createOrReplaceTempView("covid_view")  # allows using SQL on DataFrame

# Create temporary view for rows with new_deaths > 1000
spark.sql("""
CREATE OR REPLACE TEMP VIEW high_deaths AS
SELECT location, date, new_deaths
FROM covid_view
WHERE new_deaths > 1000
""")

# Left join covid_view with high_deaths to get only rows with high deaths
result = spark.sql("""
SELECT DISTINCT cv.location, cv.date, hd.new_deaths

FROM covid_view cv
RIGHT JOIN high_deaths hd
    ON cv.location = hd.location AND cv.date = hd.date

--ORDER BY location, date
ORDER BY new_deaths DESC
""")

# Show top 10 results
result.show(10)


+-------------+----------+----------+
|     location|      date|new_deaths|
+-------------+----------+----------+
|        China|2023-02-05|     47687|
|        India|2021-05-23|     28982|
|        India|2021-05-16|     27922|
|        India|2021-05-09|     26820|
|        India|2021-05-30|     26706|
|        India|2021-06-13|     23625|
|United States|2021-01-17|     23312|
|        India|2021-05-02|     23231|
|United States|2021-01-24|     22495|
|United States|2021-01-31|     22249|
+-------------+----------+----------+
only showing top 10 rows



**Expected Output**:
```
+---------+----------+----------+
| location|      date|new_deaths|
+---------+----------+----------+
|   Canada|2020-05-03|      1083|
|    China|2022-12-18|      1708|
| Colombia|2021-02-07|      2119|
|  Ecuador|2021-07-25|      8864|
|Argentina|2021-06-20|      2694|
|Argentina|2020-08-30|      2260|
|  Belgium|2020-04-12|      1938|
|   Brazil|2020-08-16|      6951|
|   Brazil|2021-05-30|     12736|
|Argentina|2020-09-27|      2456|
+---------+----------+----------+

Row count - 191529
```

### Exercise 9: Right Join with SQL
#### What to Do
Create a `high_deaths` view for rows with `new_deaths > 1000` and perform a RIGHT JOIN with `covid_view` on `location` and `date` to display all matching `location`, `date`, and `new_deaths`.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession
spark = SparkSession.builder.appName("SparkSQL8").getOrCreate()

# Read CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Filter out rows with null continent
df = df.filter(col("continent").isNotNull())

# Create temporary view to use in SQL
df.createOrReplaceTempView("covid_view")

# Create a temp view for rows with new_deaths > 1000
spark.sql("""
CREATE OR REPLACE TEMP VIEW high_deaths AS
SELECT location, date, new_deaths
FROM covid_view
WHERE new_deaths > 1000
""")

# Right join covid_view with high_deaths to get only high death rows
result = spark.sql("""
SELECT hd.location, hd.date, hd.new_deaths
FROM covid_view cv
RIGHT JOIN high_deaths hd
ON cv.location = hd.location AND cv.date = hd.date
""")

# Show top 10 results
result.show(10)


**Expected Output**:
```
+---------+----------+----------+
| location|      date|new_deaths|
+---------+----------+----------+
|Argentina|2020-07-12|      1153|
|Argentina|2020-07-19|      1374|
|Argentina|2020-07-26|      1544|
|Argentina|2020-08-02|      1456|
|Argentina|2020-08-09|      1762|
|Argentina|2020-08-16|      1662|
|Argentina|2020-08-23|      1877|
|Argentina|2020-08-30|      2260|
|Argentina|2020-09-06|      2324|
|Argentina|2020-09-13|      2446|
+---------+----------+----------+
Row count -669
```

### Exercise 10: Full Outer Join
#### What to Do
- Perform a full outer join between views of 2020 and 2021 data on `location`, showing `location`, `total_cases` (2020), and `total_cases` (2021).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession
spark = SparkSession.builder.appName("SparkSQL10").getOrCreate()

# Read CSV with header and infer schema
df = spark.read.option("header", "true").option("inferSchema", "true").csv("covid-dataset/covid-data.csv")

# Filter rows with null continent
df = df.filter(col("continent").isNotNull())

# Create temporary view for SQL queries
df.createOrReplaceTempView("covid_view")

# Temp view for max total cases in 2020 per location
spark.sql("""
CREATE OR REPLACE TEMP VIEW covid_2020 AS
SELECT location, MAX(total_cases) AS total_cases_2020
FROM covid_view
WHERE YEAR(date) = 2020
GROUP BY location
""")

# Temp view for max total cases in 2021 per location
spark.sql("""
CREATE OR REPLACE TEMP VIEW covid_2021 AS
SELECT location, MAX(total_cases) AS total_cases_2021
FROM covid_view
WHERE YEAR(date) = 2021
GROUP BY location
""")

# Full outer join to include locations from both years
result = spark.sql("""
SELECT COALESCE(c20.location, c21.location) AS location,
       c20.total_cases_2020,
       c21.total_cases_2021
FROM covid_2020 c20
FULL OUTER JOIN covid_2021 c21
ON c20.location = c21.location
""")

# Show top 10 results
result.show(10)


**Expected Output**:
```
+-------------------+----------------+----------------+
|           location|total_cases_2020|total_cases_2021|
+-------------------+----------------+----------------+
|        Afghanistan|           51848|          157902|
|            Albania|           55380|          207221|
|            Algeria|           97857|          216376|
|     American Samoa|               0|              11|
|            Andorra|            7806|           21730|
|             Angola|           17149|           71142|
|           Anguilla|              12|            1646|
|Antigua and Barbuda|             155|            4229|
|          Argentina|         1629908|         5559916|
|            Armenia|          157834|          344481|
+-------------------+----------------+----------------+
```

### Exercise 11: Self Join
#### What to Do
- Use a self join on `covid_view` to compare `total_cases` for each location between consecutive dates.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession
spark = SparkSession.builder.appName("SparkSQL11").getOrCreate()

# Read CSV with header and infer schema
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")

# Filter rows with null continent
df = df.filter(col("continent").isNotNull())

# Create temporary view for SQL queries
df.createOrReplaceTempView("covid_view")

# Calculate daily case difference by joining the table with itself on location and consecutive dates
result = spark.sql("""
SELECT c1.location,
       c1.date AS date1,
       c2.date AS date2,
       (c2.total_cases - c1.total_cases) AS case_diff
FROM covid_view c1
JOIN covid_view c2
  ON c1.location = c2.location
 AND DATEDIFF(c2.date, c1.date) = 1
WHERE c1.total_cases IS NOT NULL
  AND c2.total_cases IS NOT NULL
""")

# Show top 10 rows where the difference is not zero
result.filter("case_diff!=0").show(10)


**Expected Output**:
```
+-----------+----------+----------+---------+
|   location|     date1|     date2|case_diff|
+-----------+----------+----------+---------+
|Afghanistan|2020-02-29|2020-03-01|        1|
|Afghanistan|2020-03-14|2020-03-15|        6|
|Afghanistan|2020-03-21|2020-03-22|       17|
|Afghanistan|2020-03-28|2020-03-29|       67|
|Afghanistan|2020-04-04|2020-04-05|      183|
|Afghanistan|2020-04-11|2020-04-12|      247|
|Afghanistan|2020-04-18|2020-04-19|      387|
|Afghanistan|2020-04-25|2020-04-26|      422|
|Afghanistan|2020-05-02|2020-05-03|      841|
|Afghanistan|2020-05-09|2020-05-10|     1392|
+-----------+----------+----------+---------+
```

### Exercise 12: Window Function - Rank
#### What to Do
- Rank locations by `total_cases` within each `continent`, showing only rank 1.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession with shorter app name
spark = SparkSession.builder.appName("SparkSQL12").getOrCreate()

# Read CSV with header and infer schema
df = spark.read.option("header", "true").option("inferSchema", "true").csv("covid-dataset/covid-data.csv")

# Filter rows with non-null continent
df = df.filter(col("continent").isNotNull())

# Create temporary view for SQL queries
df.createOrReplaceTempView("covid_view")

# Find the location with the highest total_cases per continent using RANK window function
# MAX(total_cases) ensures duplicates are handled
result = spark.sql("""
SELECT continent, location, total_cases, rank
FROM (
    SELECT continent,
           location,
           MAX(total_cases) AS total_cases,
           RANK() OVER (PARTITION BY continent ORDER BY MAX(total_cases) DESC) AS rank
    FROM covid_view
    GROUP BY continent, location
) tmp
WHERE rank = 1
""")

# Display top locations per continent
result.show()


**Expected Output**:
```
+-------------+-------------+-----------+----+
|    continent|     location|total_cases|rank|
+-------------+-------------+-----------+----+
|       Africa| South Africa|    4072765|   1|
|         Asia|        China|   99373219|   1|
|       Europe|       France|   38997490|   1|
|North America|United States|  103436829|   1|
|      Oceania|    Australia|   11861161|   1|
|South America|       Brazil|   37511921|   1|
+-------------+-------------+-----------+----+
```

### Exercise 13: Window Function - Running Total
#### What to Do
- Calculate a running total of `new_cases` for each `location` over time.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession with shorter app name
spark = SparkSession.builder.appName("SparkSQL13").getOrCreate()

# Read CSV with header and infer schema
df = spark.read.option("header", "true").option("inferSchema", "true").csv("covid-dataset/covid-data.csv")

# Filter rows with non-null continent
df = df.filter(col("continent").isNotNull())

# Create temporary view for SQL queries
df.createOrReplaceTempView("covid_view")

# Calculate running total of new_cases per location using window function
# PARTITION BY location ensures each location is computed separately
# ORDER BY date ensures cumulative sum follows chronological order
# ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW accumulates from start to current row
result = spark.sql("""
SELECT location,
       date,
       new_cases,
       SUM(new_cases) OVER (
           PARTITION BY location
           ORDER BY date
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_total
FROM covid_view
WHERE new_cases IS NOT NULL
ORDER BY running_total DESC
""")

# Display top 20 rows, full content without truncation
result.show(20, False)


**Expected Output**:
```
+--------+----------+---------+-------------+
|location|date      |new_cases|running_total|
+--------+----------+---------+-------------+
|China   |2024-08-04|2087     |99373219     |
|China   |2024-07-29|0        |99371132     |
|China   |2024-07-28|2103     |99371132     |
|China   |2024-07-30|0        |99371132     |
|China   |2024-07-31|0        |99371132     |
|China   |2024-08-01|0        |99371132     |
|China   |2024-08-02|0        |99371132     |
|China   |2024-08-03|0        |99371132     |
|China   |2024-07-21|1988     |99369029     |
|China   |2024-07-22|0        |99369029     |
+--------+----------+---------+-------------+
```

### Exercise 14: Window Function - Moving Average
#### What to Do
- Calculate a 7-day moving average of `new_cases` for each `location`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession
spark = SparkSession.builder.appName("SparkSQL14").getOrCreate()

# Read CSV with header and infer schema
df = spark.read.option("header", "true").option("inferSchema", "true").csv("covid-dataset/covid-data.csv")

# Filter rows where continent is not null
df = df.filter(col("continent").isNotNull())

# Create temporary view for SQL queries
df.createOrReplaceTempView("covid_view")

# Calculate 7-day moving average of new_cases per location using window function
# PARTITION BY location ensures calculation per location
# ORDER BY date ensures correct chronological order
# ROWS BETWEEN 6 PRECEDING AND CURRENT ROW takes current and 6 previous rows (7 days)
result = spark.sql("""
SELECT location,
       date,
       new_cases,
       AVG(new_cases) OVER (
           PARTITION BY location
           ORDER BY date
           ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
       ) AS moving_avg
FROM covid_view
WHERE new_cases IS NOT NULL
ORDER BY moving_avg DESC
""")

# Display top 10 rows
result.show(10)


**Expected Output**:
```
+-------------+----------+---------+-------------+
|location     |date      |new_cases|running_total|
+-------------+----------+---------+-------------+
|United States|2023-05-14|93260    |103436829    |
|United States|2023-05-15|0        |103436829    |
|United States|2023-05-16|0        |103436829    |
|United States|2023-05-17|0        |103436829    |
|United States|2023-05-18|0        |103436829    |
|United States|2023-05-19|0        |103436829    |
|United States|2023-05-20|0        |103436829    |
|United States|2023-05-11|0        |103343569    |
|United States|2023-05-07|77165    |103343569    |
|United States|2023-05-08|0        |103343569    |
|United States|2023-05-09|0        |103343569    |
|United States|2023-05-10|0        |103343569    |
|United States|2023-05-12|0        |103343569    |
|United States|2023-05-13|0        |103343569    |
|United States|2023-04-30|86484    |103266404    |
|United States|2023-05-01|0        |103266404    |
|United States|2023-05-02|0        |103266404    |
|United States|2023-05-03|0        |103266404    |
|United States|2023-05-04|0        |103266404    |
|United States|2023-05-05|0        |103266404    |
+-------------+----------+---------+-------------+
```

### Exercise 15: Case Statement
#### What to Do
- Use a `CASE` statement to categorize `reproduction_rate` into 'Low' (<1), 'Medium' (1-1.5), and 'High' (>1.5).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession
spark = SparkSession.builder.appName("SparkSQL15").getOrCreate()

# Read CSV with header and infer schema
df = spark.read.option("header", "true").option("inferSchema", "true").csv("covid-dataset/covid-data.csv")

# Filter rows where continent is not null
df = df.filter(col("continent").isNotNull())

# Create temporary view for SQL queries
df.createOrReplaceTempView("covid_view")

# Categorize reproduction_rate into Low, Medium, High, or Unknown using CASE WHEN
result = spark.sql("""
SELECT location,
       date,
       reproduction_rate,
       CASE
           WHEN reproduction_rate < 1 THEN 'Low'
           WHEN reproduction_rate BETWEEN 1 AND 1.5 THEN 'Medium'
           WHEN reproduction_rate > 1.5 THEN 'High'
           ELSE 'Unknown'
       END AS rate_category
FROM covid_view
WHERE reproduction_rate IS NOT NULL
""")

# Show top 10 rows
result.show(10)


**Expected Output**:
```
+-----------+----------+-----------------+-------------+
|   location|      date|reproduction_rate|rate_category|
+-----------+----------+-----------------+-------------+
|Afghanistan|2020-03-29|             1.51|         High|
|Afghanistan|2020-03-30|             1.51|         High|
|Afghanistan|2020-03-31|             1.52|         High|
|Afghanistan|2020-04-01|             1.51|         High|
|Afghanistan|2020-04-02|             1.51|         High|
|Afghanistan|2020-04-03|              1.5|       Medium|
|Afghanistan|2020-04-04|             1.49|       Medium|
|Afghanistan|2020-04-05|             1.49|       Medium|
|Afghanistan|2020-04-06|             1.49|       Medium|
|Afghanistan|2020-04-07|             1.49|       Medium|
+-----------+----------+-----------------+-------------+
```

### Exercise 16: Subquery for KPI
#### What to Do
- Find the date with the highest `new_cases` for each `location` using a subquery.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession
spark = SparkSession.builder.appName("SparkSQL16").getOrCreate()

# Read CSV with header and infer schema
df = spark.read.option("header", "true").option("inferSchema", "true").csv("covid-dataset/covid-data.csv")

# Filter rows where continent is not null
df = df.filter(col("continent").isNotNull())

# Create temporary view for SQL queries
df.createOrReplaceTempView("covid_view")

# Find the date with maximum new_cases per location
result = spark.sql("""
SELECT c.location, c.date, c.new_cases
FROM covid_view c
INNER JOIN (
    SELECT location, MAX(new_cases) AS max_cases
    FROM covid_view
    GROUP BY location
) m
ON c.location = m.location AND c.new_cases = m.max_cases
""")

# Show top 10 rows
result.show(10)


**Expected Output**:
```
+-------------------+----------+---------+
|           location|      date|new_cases|
+-------------------+----------+---------+
|        Afghanistan|2021-06-27|    12314|
|            Albania|2022-01-23|    15405|
|            Algeria|2022-01-30|    14774|
|     American Samoa|2022-03-27|     1881|
|            Andorra|2022-01-16|     3480|
|             Angola|2022-01-02|    11256|
|           Anguilla|2022-01-16|      263|
|Antigua and Barbuda|2022-01-30|      627|
|          Argentina|2022-01-16|   770687|
|            Armenia|2022-02-06|    23524|
+-------------------+----------+---------+
```

### Exercise 17: Compare SQL and DataFrame API
#### What to Do
- Calculate the maximum `new_deaths` per `continent` using both Spark SQL and DataFrame API.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession
spark = SparkSession.builder.appName("SparkSQL17").getOrCreate()

# Read CSV with header and infer schema
df = spark.read.option("header", "true").option("inferSchema", "true").csv("covid-dataset/covid-data.csv")

# Filter rows where continent is not null
df = df.filter(col("continent").isNotNull())

# Create temporary view for SQL queries
df.createOrReplaceTempView("covid_view")

# Using Spark SQL: max new_deaths per continent
sql_result = spark.sql("""
SELECT continent, MAX(new_deaths) AS max_new_deaths
FROM covid_view
WHERE continent IS NOT NULL
GROUP BY continent
""")
sql_result.show()

# Using DataFrame API: equivalent computation
df_result = df.groupBy("continent").agg(max("new_deaths").alias("max_new_deaths"))
df_result.show()


**Expected Output**:
```
+-------------+--------------+
|    continent|max_new_deaths|
+-------------+--------------+
|       Europe|          9723|
|       Africa|          4027|
|North America|         23312|
|South America|         21094|
|      Oceania|          1161|
|         Asia|         47687|
+-------------+--------------+

+-------------+--------------+
|    continent|max_new_deaths|
+-------------+--------------+
|       Europe|          9723|
|       Africa|          4027|
|North America|         23312|
|South America|         21094|
|      Oceania|          1161|
|         Asia|         47687|
+-------------+--------------+
```

### Exercise 18: Cross Join for Combinations
#### What to Do
- Create a view of distinct continents and perform a cross join with a view of years (2020, 2021).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession
spark = SparkSession.builder.appName("SparkSQL18").getOrCreate()

# Read CSV and filter non-null continents
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")
df = df.filter(col("continent").isNotNull())

# Create temporary view for SQL
df.createOrReplaceTempView("covid_view")

# Create a temp view with distinct continents
spark.sql("""
CREATE OR REPLACE TEMP VIEW continents AS
SELECT DISTINCT continent
FROM covid_view
WHERE continent IS NOT NULL
""")

# Create a DataFrame of years and view
years_df = spark.createDataFrame([(2020,), (2021,)], ["year"])
years_df.createOrReplaceTempView("years")

# Cross join continents and years
result = spark.sql("""
SELECT c.continent, y.year
FROM continents c
CROSS JOIN years y
""")
result.show()


**Expected Output**:
```
+-------------+----+
|    continent|year|
+-------------+----+
|       Europe|2020|
|       Africa|2020|
|North America|2020|
|South America|2020|
|      Oceania|2020|
|         Asia|2020|
|       Europe|2021|
|       Africa|2021|
|North America|2021|
|South America|2021|
|      Oceania|2021|
|         Asia|2021|
+-------------+----+
```

### Exercise 19: KPI - Case Fatality Rate
#### What to Do
- Calculate the case fatality rate (`total_deaths` / `total_cases`) per `location` for the latest date.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession
spark = SparkSession.builder.appName("SparkSQL19").getOrCreate()

# Read CSV and filter non-null continents
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")
df = df.filter(col("continent").isNotNull())

# Create temporary view for SQL
df.createOrReplaceTempView("covid_view")

# Calculate latest fatality rate per location
result = spark.sql("""
SELECT distinct c.location, c.date,
       COALESCE(c.total_deaths / NULLIF(c.total_cases, 0), 0) AS fatality_rate
FROM covid_view c
INNER JOIN (
    SELECT location, MAX(date) AS max_date
    FROM covid_view
    GROUP BY location
) m ON c.location = m.location AND c.date = m.max_date
WHERE c.total_cases IS NOT NULL AND c.total_deaths IS NOT NULL
""")

# Show top 10 results
result.show(10)


**Expected Output**:
```
+-------------------+----------+--------------------+
|           location|      date|       fatality_rate|
+-------------------+----------+--------------------+
|        Afghanistan|2024-08-04| 0.03400307804807537|
|            Albania|2024-08-04|0.010759684462179933|
|            Algeria|2024-08-04|  0.0252848728039715|
|     American Samoa|2024-08-04|0.004067472185668142|
|            Andorra|2024-08-04| 0.00331146516713527|
|             Angola|2024-08-04| 0.01802178989774937|
|           Anguilla|2024-08-04|0.003073770491803...|
|Antigua and Barbuda|2024-08-04| 0.01603338458159455|
|          Argentina|2024-08-04|0.012935370764198931|
|            Armenia|2024-08-04|0.019406420458439926|
+-------------------+----------+--------------------+
```

### Exercise 20: Save Query Results
#### What to Do
- Use Spark SQL to find locations with `total_cases` > 500000 and save the results (`location`, `total_cases`, `date`) to a CSV file.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize SparkSession
spark = SparkSession.builder.appName("SparkSQL20").getOrCreate()

# Read CSV and filter non-null continents
df = spark.read.option("header","true").option("inferSchema","true").csv("covid-dataset/covid-data.csv")
df = df.filter(col("continent").isNotNull())

# Create temporary view for SQL
df.createOrReplaceTempView("covid_view")

# Select locations with total_cases > 500000
result = spark.sql("""
SELECT location, total_cases, date
FROM covid_view
WHERE total_cases > 500000
""")

# Save result to CSV
result.write.option("header","true").mode("overwrite").csv("output/high_cases_sql")

# Show top 5 results
result.show(5)


**Expected Output**:
```
+-------------+-----------+----------+
|     location|total_cases|      date|
+-------------+-----------+----------+
|    Argentina|    5001234|2021-06-15|
|    Australia|   10585719|2022-01-20|
|      Austria|    636634|2021-11-15|
|   Bangladesh|    798830|2021-07-10|
|      Belgium|   1234567|2021-12-01|
+-------------+-----------+----------+
```

## Finish Up
- Save as `solutions/04_solution.ipynb`.
- Re-run cells if errors occur.
- Great job learning Spark SQL and joins!